# Feature engineering for transit ridership forecasting

This notebook builds the feature set used to predict monthly transit ridership.
We create lag features, rolling means, year-over-year changes, and seasonal indicators,
then examine correlations to confirm their predictive value.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.insert(0, '..')

from src.data_loader import (
    load_or_fetch_ridership, preprocess_ridership, engineer_features
)
from src.model import NUMERICAL_FEATURES, TARGET

pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')
plt.rcParams['figure.dpi'] = 120
sns.set_style('whitegrid')

## 1. Load and preprocess transit data

In [ ]:
df_raw = load_or_fetch_ridership('../data', limit=100000)
print(f'Raw shape: {df_raw.shape}')
print(f'Columns: {list(df_raw.columns)}')

df = preprocess_ridership(df_raw)
print(f'\nAfter preprocessing: {df.shape}')
df.head()

## 2. Create lag features

Lag features capture the autoregressive nature of time-series data.
We use 1-month, 3-month, and 12-month lags. The 12-month lag is
particularly important because transit ridership follows strong annual cycles.

In [ ]:
df = engineer_features(df)

# Display lag features alongside the target
lag_cols = ['ridership', 'lag_1m', 'lag_3m', 'lag_12m']
available_lags = [c for c in lag_cols if c in df.columns]
print('Lag features (first 15 rows where all are available):')
df[available_lags].dropna().head(15)

In [ ]:
# Visualize lag relationships
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, lag_col in zip(axes, ['lag_1m', 'lag_3m', 'lag_12m']):
    if lag_col in df.columns:
        valid = df.dropna(subset=[lag_col, 'ridership'])
        ax.scatter(valid[lag_col], valid['ridership'], alpha=0.5, s=20)
        ax.set_xlabel(lag_col)
        ax.set_ylabel('ridership')
        corr = valid[lag_col].corr(valid['ridership'])
        ax.set_title(f'{lag_col} vs ridership (r={corr:.3f})')

plt.tight_layout()
plt.show()

## 3. Rolling mean features

Rolling means smooth out short-term noise and capture the underlying trend.
We compute 3-month, 6-month, and 12-month rolling windows.

In [ ]:
rolling_cols = ['ridership', 'rolling_mean_3m', 'rolling_mean_6m', 'rolling_mean_12m']
available_rolling = [c for c in rolling_cols if c in df.columns]

if 'date' in df.columns and len(available_rolling) > 1:
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(df['date'], df['ridership'], alpha=0.4, label='Actual ridership', linewidth=1)
    for col in available_rolling[1:]:
        ax.plot(df['date'], df[col], label=col, linewidth=1.5)
    ax.set_xlabel('Date')
    ax.set_ylabel('Ridership')
    ax.set_title('Ridership with rolling mean overlays')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print('Rolling means computed:')
    print(df[available_rolling].describe())

## 4. Year-over-year change

The YoY change measures the percentage difference from the same month a year ago.
This feature captures whether ridership is growing or declining relative to
the prior year's seasonal baseline.

In [ ]:
if 'yoy_change' in df.columns:
    valid_yoy = df.dropna(subset=['yoy_change'])
    print(f'YoY change statistics:')
    print(valid_yoy['yoy_change'].describe())

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Distribution of YoY change
    axes[0].hist(valid_yoy['yoy_change'], bins=30, edgecolor='black', alpha=0.7)
    axes[0].axvline(0, color='red', linestyle='--', linewidth=1.5)
    axes[0].set_xlabel('Year-over-year change (%)')
    axes[0].set_ylabel('Frequency')
    axes[0].set_title('Distribution of YoY change')

    # Time-series of YoY change
    if 'date' in valid_yoy.columns:
        axes[1].plot(valid_yoy['date'], valid_yoy['yoy_change'], linewidth=1)
        axes[1].axhline(0, color='red', linestyle='--', linewidth=1)
        axes[1].set_xlabel('Date')
        axes[1].set_ylabel('YoY change (%)')
        axes[1].set_title('Year-over-year change over time')

    plt.tight_layout()
    plt.show()

## 5. Seasonal indicators

Month and quarter encode the seasonal cycle. Transit ridership in Calgary
typically peaks during school and work months (September-November) and dips
during summer vacation and holiday periods.

In [ ]:
seasonal_cols = ['month', 'quarter']
available_seasonal = [c for c in seasonal_cols if c in df.columns]

if 'month' in df.columns and 'ridership' in df.columns:
    monthly_avg = df.groupby('month')['ridership'].agg(['mean', 'std']).reset_index()
    month_labels = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                    'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Monthly average ridership
    axes[0].bar(monthly_avg['month'], monthly_avg['mean'],
                yerr=monthly_avg['std'], capsize=3, color='steelblue', alpha=0.8)
    axes[0].set_xticks(range(1, 13))
    axes[0].set_xticklabels(month_labels, rotation=45)
    axes[0].set_ylabel('Average ridership')
    axes[0].set_title('Average ridership by month')

    # Quarterly boxplot
    if 'quarter' in df.columns:
        df.boxplot(column='ridership', by='quarter', ax=axes[1])
        axes[1].set_xlabel('Quarter')
        axes[1].set_ylabel('Ridership')
        axes[1].set_title('Ridership distribution by quarter')
        plt.suptitle('')

    plt.tight_layout()
    plt.show()
else:
    print('Seasonal indicators:', available_seasonal)
    print(df[available_seasonal].describe())

## 6. Feature correlation matrix

We examine how all engineered features correlate with the target variable
and with each other. High inter-feature correlation (multicollinearity)
is expected between rolling means of different windows, but tree-based
models handle this well. Ridge regression uses regularization to manage it.

In [ ]:
all_features = [c for c in NUMERICAL_FEATURES if c in df.columns] + [TARGET]
available = [c for c in all_features if c in df.columns]

corr_df = df[available].dropna()
print(f'Rows available for correlation (after dropping NaN from lags): {len(corr_df)}')

corr_matrix = corr_df.corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='RdBu_r', center=0, ax=ax, linewidths=0.5)
ax.set_title('Feature correlation matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation with target, sorted
if TARGET in corr_matrix.columns:
    target_corr = corr_matrix[TARGET].drop(TARGET).sort_values(ascending=False)
    print('Feature correlations with ridership (sorted):\n')
    for feat, corr_val in target_corr.items():
        bar = '+' * int(abs(corr_val) * 40)
        sign = '+' if corr_val >= 0 else '-'
        print(f'  {feat:25s}  {corr_val:+.3f}  {sign}{bar}')

## 7. Final feature set summary

We confirm the feature set that will be passed to the modeling notebook.
The `model.py` module defines `NUMERICAL_FEATURES` as the canonical list.

In [ ]:
from src.model import prepare_model_data

X, y, _, used_features = prepare_model_data(df)
print(f'Feature matrix shape: {X.shape}')
print(f'Target vector length: {len(y)}')
print(f'\nFeatures used ({len(used_features)}):')
for i, feat in enumerate(used_features, 1):
    print(f'  {i}. {feat}')

print(f'\nTarget statistics:')
print(y.describe())

In [ ]:
# Check for any remaining NaN values in the final feature matrix
nan_counts = X.isnull().sum()
print('NaN counts per feature after prepare_model_data():')
print(nan_counts)
print(f'\nTotal NaN cells: {nan_counts.sum()}')
print(f'Ready for modeling: {nan_counts.sum() == 0}')

## Summary

The feature engineering pipeline produces 10 features from the raw ridership data:

- **Lag features** (lag_1m, lag_3m, lag_12m) capture autoregressive signal
- **Rolling means** (3m, 6m, 12m) smooth short-term volatility
- **YoY change** measures growth/decline relative to the prior year
- **Seasonal indicators** (month, quarter, year) encode cyclical patterns

The 12-month lag and rolling means show the highest correlation with the target,
consistent with the strong annual seasonality in Calgary transit data.
Next step: train Ridge, Random Forest, and XGBoost models in `03_modeling.ipynb`.